## Kelley hu4d5-5 VL-R66G charging step error analysis

In this notebook we will use a random forest model to find the most energetically influential degrees of freedom for the VL-R66G charging step TI production run. Next, we will compare the sampling of these DOF during TI production to a free energy profile derived from end state GaMD sampling. We will attempt to  correct any inaccurate sampling in the TI data and find the estimated ddG before and after the correction. 

In [1]:
import os
os.chdir("..")
from common_functions import *

### Ingesting original TI lambda production data

In [2]:
os.chdir("./TI_data/VL-R66G")
geom_dvdls_crg = pd.read_csv("R66G_crg_bound.csv")
geom_dvdls_crg_ub = pd.read_csv("R66G_crg_unbound.csv")
geom_dvdls_vdw = pd.read_csv("R66G_vdw_bound.csv")
geom_dvdls_vdw_ub = pd.read_csv("R66G_vdw_unbound.csv")


### Initial VL-R66G TI ddG estimate:

In [3]:
orig_vdw_dGs = np.array(bootstrap_df(geom_dvdls_vdw, 1000, 200))
orig_vdw_ub_dGs = np.array(bootstrap_df(geom_dvdls_vdw_ub, 1000, 200))
orig_crg_dGs = np.array(bootstrap_df(geom_dvdls_crg, 1000, 200))
orig_crg_ub_dGs = np.array(bootstrap_df(geom_dvdls_crg_ub, 1000, 200))

In [4]:
total_ddGs = print_summary_crg(orig_crg_dGs, orig_crg_ub_dGs, orig_vdw_dGs, orig_vdw_ub_dGs)
empirical_value = 0.22
orig_error = abs((np.mean(total_ddGs)) - empirical_value)

print(f"\nEmpirical value: {empirical_value} kcal/mol")
print(f"Original error: {round(orig_error, 4)} kcal/mol")

Charging step ddG: -0.8542 +/- 0.8496
Lower CI: -1.7038
Upper CI: -0.0106

Vdw step ddG: 0.1512 +/- 0.6432
Lower CI: -0.4655
Upper CI: 0.7944

Total ddG: -0.703 +/- 1.0762
Lower CI: -1.7437
Upper CI: 0.3732

Empirical value: 0.22 kcal/mol
Original error: 0.923 kcal/mol


### Charging step RF model

#### Splitting data into independent/dependent variables for random forest model

See our methods/supplemental methods section for our process to choose the input features.

In [5]:
X = geom_dvdls_crg.drop(
    ["weight_dvdl", "dvdl", "Run", "Lambda", "#Frame", "R66_T69", "R66_G68", "R421_chi1"
    ], axis=1)

X_scl = pd.DataFrame(StandardScaler().fit_transform(X))
X_scl.columns = X.columns
Y = geom_dvdls_crg["weight_dvdl"]


#### Checking for cross-correlation among independent variables

In [6]:
absCorr = abs(X_scl.corr())
for i in absCorr.columns:
    for j in absCorr.index:
        cor = absCorr.loc[i, j]
        if abs(cor) > 0.5 and i != j:
            print(i, j)
            print(cor)
            

#### Using random forest model to identify the most influential degrees of freedom

We run our model 25 times, then sort the results by the mean of feature importance across the 25 iterations. The model found that VL-N30 side chain rotamers chi1 and chi2 were the most influential nearby DOF on TI DV/DL.

In [7]:
rfeDefault = RFE(estimator=DecisionTreeRegressor(max_depth=5, random_state=42), n_features_to_select=0.75, step=0.05)
rfDefault = RandomForestRegressor(
    max_depth=10, n_estimators=200, oob_score=True, max_features=0.6, min_samples_leaf = 7, min_samples_split=14, random_state=42,
    n_jobs=-1
)

pipelineDefault_rf = Pipeline([
    ('feature_scaling', StandardScaler()),
    ('feature_selection', rfeDefault),
    ('regression_model', rfDefault)
])


imps = benchmark_model(pipelineDefault_rf, X_scl, Y, geom_dvdls_crg["Lambda"])
imps[["Mean", "Median"]].sort_values(by="Mean", ascending=False)[:15]

Avg. training r2: 
0.7394
Training r2 std dev: 
0.0004
Avg. test r2: 
0.6592
Testing r2 std dev: 
0.0046


,Mean,Median
N385_chi1,0.243912,0.252100
N385_chi2,0.133807,0.127892
D383_chi1,0.101620,0.101337
R66_D28,0.082364,0.081956
T427_chi1,0.074436,0.074156
S422_chi1,0.067838,0.073362
S407_chi1,0.067217,0.067412
D425_chi2,0.057261,0.057207
V384_chi1,0.044636,0.045734
T386_chi1,0.038542,0.034911


### Comparing the TI rotamers with the GaMD pmf (bound state)

The RF model tells us that the most energetically influential features to DV/DL are "N385_chi1" and "N385_chi2", aka VL-N30 side chain rotamers chi1 and chi2. We used GaMD on the WT hu4d5-5 complex to obtain enhanced sampling data for this end state. Then, we used PyReweighting-2D to generate a 2-D PMF of VL-N30 chi1 and VL-N30 chi2 to give us an idea of the ideal sampling of these two energetically influential features. Finally, we plot the VL-N30 rotamers explored during TI overlaid onto the GaMD pmf to probe the quality of sampling during TI.

Next, we will compare the proportion of TI sampling in each of the three predominant microstates to the relative areas of the microstates in the GaMD end state free energy profile.

In [8]:
os.chdir("../../gamd_pmfs/VL-R66G")
n385_pmf = get_pmf_2d(
    "./pmf-c2-N165_c1c2_bound_300ns.dat.xvg"
)

geom_dvdls_crg["in_pmf_cont"] = rot_in_pmf_cont(geom_dvdls_crg, "N385_chi1", "N385_chi2", n385_pmf, 10)

In [9]:
n385_pmf.loc[n385_pmf["X"] < 0, "X"] += 360
geom_dvdls_crg.loc[geom_dvdls_crg["N385_chi1"] < 0, "N385_chi1"] += 360

fig = plot_pmf_with_TI_2d_fig(n385_pmf, geom_dvdls_crg[geom_dvdls_crg["in_pmf_cont"]], "N385_chi1", "N385_chi2")
fig.update_xaxes(
    title="VL-N30 chi1"
).update_yaxes(
    title="VL-N30 chi2"
).update_layout(width=1000, height=700)
fig.show(renderer="svg")

In [10]:
a1 = n385_pmf[n385_pmf["X"] < 100]["pop"].sum()
a2 = n385_pmf[(n385_pmf["X"] > 100) & (n385_pmf["X"] < 220)]["pop"].sum()
a3 = n385_pmf[(n385_pmf["X"] > 220) & (n385_pmf["X"] < 360)]["pop"].sum()

print(f"VL-N30 chi1 < 100 degrees proportion: {round(100 * a1/(a1 + a2 + a3), 4)}%")
print(f"100 < VL-N30 chi1 < 220 degrees proportion: {round(100 * a2/(a1 + a2 + a3), 4)}%")
print(f"220 < VL-N30 chi1 < 360 degrees proportion: {round(100 * a3/(a1 + a2 + a3), 4)}%")

a1_prop = a1/(a1 + a2)
a2_prop = a2/(a1 + a2)

VL-N30 chi1 < 100 degrees proportion: 50.6516%
100 < VL-N30 chi1 < 220 degrees proportion: 49.1268%
220 < VL-N30 chi1 < 360 degrees proportion: 0.2216%


### Checking sampling of the two predominant microstates

We will neglect the one to the far right since it is only 0.22% of the total area of the free energy profile. 

The first microstate (leftmost) has insufficient sampling in many lambdas. We will run more TI lambda production with starting rotamers specifically in this microstate to boost sampling.

In [11]:
st1_crg = geom_dvdls_crg[
    (geom_dvdls_crg["in_pmf_cont"]) &
    (geom_dvdls_crg["N385_chi1"] < 100) 
]

st2_crg = geom_dvdls_crg[
    (geom_dvdls_crg["in_pmf_cont"]) &
    (geom_dvdls_crg["N385_chi1"] > 100) &
    (geom_dvdls_crg["N385_chi1"] < 220)      
]

print(st1_crg.groupby("Lambda").count()["weight_dvdl"])
print(st2_crg.groupby("Lambda").count()["weight_dvdl"])

Lambda
1     393
2     543
3     572
4     567
5     478
6     264
7     617
8     746
9     827
10    500
11    271
12    309
Name: weight_dvdl, dtype: int64
Lambda
2       6
5      15
6     159
7     105
8       4
9      60
10    366
11    589
12    475
Name: weight_dvdl, dtype: int64


### Ingesting boosted TI lambda production sampling on the right-hand predominant state

In [12]:
os.chdir("../../TI_data/VL-R66G")
geom_dvdls_boost = pd.read_csv("R66G_crg_boosted.csv")

geom_dvdls_boost.loc[geom_dvdls_boost["N385_chi1"] < 0, "N385_chi1"] += 360
geom_dvdls_boost["in_pmf_cont"] = rot_in_pmf_cont(geom_dvdls_boost, "N385_chi1", "N385_chi2", n385_pmf, 10)

#### Checking the sampling on the microstates to make sure it is sufficient

In [13]:
st2_missing = geom_dvdls_boost[
    (geom_dvdls_boost["in_pmf_cont"]) & 
    (geom_dvdls_boost["N385_chi1"] > 100) &
    (geom_dvdls_boost["N385_chi1"] < 220) & 
    ((geom_dvdls_boost["Lambda"] <= 5) | 
     (geom_dvdls_boost["Lambda"] == 8) | 
     (geom_dvdls_boost["Lambda"] == 9))
]

st2boost = pd.concat([st2_crg, st2_missing])

print(st1_crg.groupby("Lambda").count()["weight_dvdl"])
print(st2boost.groupby("Lambda").count()["weight_dvdl"])


Lambda
1     393
2     543
3     572
4     567
5     478
6     264
7     617
8     746
9     827
10    500
11    271
12    309
Name: weight_dvdl, dtype: int64
Lambda
1     159
2     185
3     542
4     596
5     713
6     159
7     105
8     973
9     982
10    366
11    589
12    475
Name: weight_dvdl, dtype: int64


In [14]:
st1boost_dGs = bootstrap_df(st1_crg, 1000, 200)
st2boost_dGs = bootstrap_df(st2boost, 1000, 200)

### Comparing TI sampling of the two microstates to the GaMD free energy profile areas

In [15]:
print("GaMD pmf left hand state proportion: ")
print(round(a1_prop, 4))
print("GaMD pmf middle state proportion: ")
print(round(a2_prop, 4))

ti1 = len(st1_crg)
ti2 = len(st2_crg)

print("TI left hand state proportion: ")
print(round(ti1/(ti1 + ti2), 4))
print("TI middle hand state proportion: ")
print(round(ti2/(ti1 + ti2), 4))


GaMD pmf left hand state proportion: 
0.5076
GaMD pmf middle state proportion: 
0.4924
TI left hand state proportion: 
0.7738
TI middle hand state proportion: 
0.2262


#### Computing corrected charging step bound state dG

This is simply computing the dG of each state then combining them in a weighted sum, weighted by the microstates in the GaMD pmf.

In [16]:
corr_crg_dGs = a1_prop * st1boost_dGs + a2_prop * st2boost_dGs 

### Comparing GaMD with TI, unbound state

In [17]:
os.chdir("../../gamd_pmfs/VL-R66G")
n30_pmf_ub = get_pmf_2d(
    "./pmf-c2-N30c1c2_unbound_300ns.dat.xvg"
)

n30_pmf_ub.loc[n30_pmf_ub["X"] < 0, "X"] += 360
geom_dvdls_crg_ub.loc[geom_dvdls_crg_ub["N250_chi1"] < 0, "N250_chi1"] += 360
geom_dvdls_crg_ub["in_pmf_cont"] = rot_in_pmf_cont(geom_dvdls_crg_ub, "N250_chi1", "N250_chi2", n30_pmf_ub, 10)


In [18]:
fig = plot_pmf_with_TI_2d_fig(n30_pmf_ub, geom_dvdls_crg_ub[geom_dvdls_crg_ub["in_pmf_cont"]], "N250_chi1", "N250_chi2")
fig.update_xaxes(
    title="VL-N30 chi1"
).update_yaxes(
    title="VL-N30 chi2"
).update_layout(width=1000, height=700)
fig.show(renderer="svg")

### Checking sampling of the two microstates

The righthand microstate has insufficient sampling in lambdas 2 and 12 and needs boosted TI lambda production sampling.

In [19]:
st1ub = geom_dvdls_crg_ub[(geom_dvdls_crg_ub["N250_chi1"] < 100) & (geom_dvdls_crg_ub["in_pmf_cont"])]
st2ub = geom_dvdls_crg_ub[
    (geom_dvdls_crg_ub["N250_chi1"] > 100) & 
    (geom_dvdls_crg_ub["N250_chi1"] < 220) &
    (geom_dvdls_crg_ub["in_pmf_cont"])]

print(st1ub.groupby("Lambda").count()["weight_dvdl"])
print(st2ub.groupby("Lambda").count()["weight_dvdl"])

Lambda
1     428
2     487
3     595
4     585
5     574
6     540
7     388
8     409
9     528
10    368
11    404
12    538
Name: weight_dvdl, dtype: int64
Lambda
1     245
2     111
3      39
4     110
5     198
6     188
7     214
8     306
9     203
10    245
11    231
12     95
Name: weight_dvdl, dtype: int64


#### Ingesting more sampling for unbound state

In [20]:
os.chdir("../../TI_data/VL-R66G")
geom_dvdls_boost_ub = pd.read_csv("R66G_crg_boost_ub.csv")
geom_dvdls_boost_ub.loc[geom_dvdls_boost_ub["N250_chi1"] < 0, "N250_chi1"] += 360

geom_dvdls_boost_ub["in_pmf_cont"] = rot_in_pmf_cont(geom_dvdls_boost_ub, "N250_chi1", "N250_chi2", n30_pmf_ub, 10)


#### Ingesting boosted sampling in righthand state

In [21]:
st2_missing_ub = geom_dvdls_boost_ub[
    (geom_dvdls_boost_ub["in_pmf_cont"]) & 
    (geom_dvdls_boost_ub["N250_chi1"] > 100) &
    (geom_dvdls_boost_ub["N250_chi1"] < 220) & 
    ((geom_dvdls_boost_ub["Lambda"] == 3) | 
     (geom_dvdls_boost_ub["Lambda"] == 12)) 
]

st2boost_ub = pd.concat([st2ub, st2_missing_ub])

print(st1ub.groupby("Lambda").count()["weight_dvdl"])
print(st2boost_ub.groupby("Lambda").count()["weight_dvdl"])


Lambda
1     428
2     487
3     595
4     585
5     574
6     540
7     388
8     409
9     528
10    368
11    404
12    538
Name: weight_dvdl, dtype: int64
Lambda
1      245
2      111
3      564
4      110
5      198
6      188
7      214
8      306
9      203
10     245
11     231
12    1012
Name: weight_dvdl, dtype: int64


### Comparing TI sampling of the three microstates to the GaMD free energy profile areas

In [22]:
a1ub = n30_pmf_ub[(n30_pmf_ub["X"] < 100)]["pop"].sum()
a2ub = n30_pmf_ub[(n30_pmf_ub["X"] > 100) & (n30_pmf_ub["X"] < 220)]["pop"].sum()

a1_prop_ub = a1ub/(a1ub + a2ub)
a2_prop_ub = a2ub/(a1ub + a2ub)

print("GaMD pmf left hand state proportion: ")
print(round(a1_prop_ub, 4))
print("GaMD pmf right hand state proportion: ")
print(round(a2_prop_ub, 4))

ti1_ub = len(st1ub)
ti2_ub = len(st2ub)

print()
print("TI original, unboosted proportions")
print("TI left hand state proportion: ")
print(round(ti1_ub/(ti1_ub + ti2_ub ), 4))
print("TI right hand state proportion: ")
print(round(ti2_ub/(ti1_ub + ti2_ub ), 4))


GaMD pmf left hand state proportion: 
0.742
GaMD pmf right hand state proportion: 
0.258

TI original, unboosted proportions
TI left hand state proportion: 
0.7279
TI right hand state proportion: 
0.2721


#### Recomputing unbound charging step dG and corrected charging step ddG

In [23]:
st1boost_ub_dGs = np.array(bootstrap_df(st1ub, 1000, 200))
st2boost_ub_dGs = np.array(bootstrap_df(st2boost_ub, 1000, 200))


In [24]:
corr_crg_ub_dGs = a1_prop_ub * st1boost_ub_dGs + a2_prop_ub * st2boost_ub_dGs 

#### Charging step correction results

With the correction based on the VL-N30 chi1/chi2 rotamer sampling, there is slight change in the charging step ddG (~0.1-0.2 kcal/mol increase)

Next, we will incorporate the corrections from the vdw step (found in the `R66Gvdw_example.ipynb` notebook).

In [25]:
print("Values after crg step correction, no vdw step correction")

corr_ddGs = print_summary_crg(corr_crg_dGs, corr_crg_ub_dGs, orig_vdw_dGs, orig_vdw_ub_dGs)

Values after crg step correction, no vdw step correction
Charging step ddG: -1.0133 +/- 0.6596
Lower CI: -1.6729
Upper CI: -0.3755

Vdw step ddG: 0.1512 +/- 0.6432
Lower CI: -0.4655
Upper CI: 0.7944

Total ddG: -0.8622 +/- 0.8947
Lower CI: -1.7569
Upper CI: 0.0269


#### Original results for comparison 

In [26]:
total_ddGs = print_summary_crg(orig_crg_dGs, orig_crg_ub_dGs, orig_vdw_dGs, orig_vdw_ub_dGs)


Charging step ddG: -0.8542 +/- 0.8496
Lower CI: -1.7038
Upper CI: -0.0106

Vdw step ddG: 0.1512 +/- 0.6432
Lower CI: -0.4655
Upper CI: 0.7944

Total ddG: -0.703 +/- 1.0762
Lower CI: -1.7437
Upper CI: 0.3732


#### Incorporating vdw step correction (bound state)

We will concisely reproduce the correction here, but for more detail please see the `R66Gvdw_example.ipynb` notebook. 

In [27]:
# importing GaMD pmf for bound state VL-F71 chi2, VL-D28 chi1 (most influential rotamers according to vdw step RF model)
os.chdir("../../gamd_pmfs/VL-R66G")
f71c2_d28c1 = get_pmf_2d(
    "pmf-c2-F71c2_D28c1_bound.dat.xvg"
)
# shifting coordinates for simplicity (-180, 180) --> (0, 360) 
f71c2_d28c1.loc[f71c2_d28c1["Y"] < 0, "Y"] += 360

# importing GaMD pmf for unbound state VL-F71 chi2, VL-D28 chi1 (most influential rotamers according to vdw step RF model)
f71c2_d28c1_ub = get_pmf_2d(
    "pmf-c2-F71c2_D28c1_unbound.dat.xvg"
)
# shifting coordinates for simplicity (-180, 180) --> (0, 360) 
f71c2_d28c1_ub.loc[f71c2_d28c1_ub["Y"] < 0, "Y"] += 360
geom_dvdls_vdw.loc[geom_dvdls_vdw["D383_chi1"] < 0, "D383_chi1"] += 360

# filtering TI sampling to match GaMD pmf
geom_dvdls_vdw["in_pmf_cont"] = rot_in_pmf_cont(geom_dvdls_vdw, "F426_chi2", "D383_chi1", f71c2_d28c1, 10)

In [28]:
st1_vdw = geom_dvdls_vdw[
    (geom_dvdls_vdw["in_pmf_cont"]) & 
    (geom_dvdls_vdw["D383_chi1"] < 100) 
]

st2_vdw = geom_dvdls_vdw[
    (geom_dvdls_vdw["in_pmf_cont"]) & 
    (geom_dvdls_vdw["D383_chi1"] > 100) &
    (geom_dvdls_vdw["D383_chi1"] < 220) 
]

st3_vdw = geom_dvdls_vdw[
    (geom_dvdls_vdw["in_pmf_cont"]) & 
    (geom_dvdls_vdw["D383_chi1"] > 220) 
]

print(st1_vdw.groupby("Lambda").count()["weight_dvdl"])
print(st2_vdw.groupby("Lambda").count()["weight_dvdl"])
print(st3_vdw.groupby("Lambda").count()["weight_dvdl"])

Lambda
1      99
4      39
5     137
7     192
8     197
12     66
Name: weight_dvdl, dtype: int64
Lambda
1      95
2     188
3     153
4     186
5     211
6     332
7       8
8      10
9     133
10     16
Name: weight_dvdl, dtype: int64
Lambda
1     168
2     155
3     182
4     106
7     149
8     175
9     228
10    347
11    386
12    317
Name: weight_dvdl, dtype: int64


#### Ingesting more sampling (vdw bound state)

In [29]:
os.chdir("../../TI_data/VL-R66G")
geom_dvdls_vdw_boost = pd.read_csv("R66G_vdw_boosted.csv")

geom_dvdls_vdw_boost.loc[geom_dvdls_vdw_boost["D383_chi1"] < 0, "D383_chi1"] += 360
geom_dvdls_vdw_boost["in_pmf_cont"] = rot_in_pmf_cont(geom_dvdls_vdw_boost, "F426_chi2", "D383_chi1", f71c2_d28c1, 10)

st1_vdw_missing = geom_dvdls_vdw_boost[
    (geom_dvdls_vdw_boost["in_pmf_cont"]) & 
    (geom_dvdls_vdw_boost["D383_chi1"] < 100) &
    (geom_dvdls_vdw_boost["Lambda"].isin([1, 2, 3, 4, 6, 9, 10, 11, 12]))
]

st2_vdw_missing = geom_dvdls_vdw_boost[
    (geom_dvdls_vdw_boost["in_pmf_cont"]) & 
    (geom_dvdls_vdw_boost["D383_chi1"] > 100) &
    (geom_dvdls_vdw_boost["D383_chi1"] < 220) &
    (geom_dvdls_vdw_boost["Lambda"].isin([1, 7, 8, 10, 11, 12]))

]

st3_vdw_missing = geom_dvdls_vdw_boost[
    (geom_dvdls_vdw_boost["in_pmf_cont"]) & 
    (geom_dvdls_vdw_boost["D383_chi1"] > 220) &
    (geom_dvdls_vdw_boost["Lambda"].isin([5, 6]))
]

st1_vdw_boost = pd.concat([st1_vdw, st1_vdw_missing])
st2_vdw_boost = pd.concat([st2_vdw, st2_vdw_missing])
st3_vdw_boost = pd.concat([st3_vdw, st3_vdw_missing])

print(st1_vdw_boost.groupby("Lambda").count()["weight_dvdl"])
print(st2_vdw_boost.groupby("Lambda").count()["weight_dvdl"])
print(st3_vdw_boost.groupby("Lambda").count()["weight_dvdl"])

Lambda
1     933
2       4
3     235
4      39
5     137
7     192
8     197
9     172
10    460
11    231
12    474
Name: weight_dvdl, dtype: int64
Lambda
1     235
2     188
3     153
4     186
5     211
6     332
7     458
8     749
9     133
10    507
11    385
12    253
Name: weight_dvdl, dtype: int64
Lambda
1     168
2     155
3     182
4     106
5     611
6     801
7     149
8     175
9     228
10    347
11    386
12    317
Name: weight_dvdl, dtype: int64


#### Filling in remaining missing microstate 1 values with the other two microstates

In [30]:
for_st1_vdw = pd.concat([st2_vdw_boost, st3_vdw_boost])
st1_vdw_missing_postBoost = for_st1_vdw[for_st1_vdw["Lambda"].isin([2, 4, 6])]
st1_vdw_for_dG = pd.concat([st1_vdw_boost, st1_vdw_missing_postBoost])

print(st1_vdw_for_dG.groupby("Lambda").count()["weight_dvdl"])
print(st2_vdw_boost.groupby("Lambda").count()["weight_dvdl"])
print(st3_vdw_boost.groupby("Lambda").count()["weight_dvdl"])


Lambda
1      933
2      347
3      235
4      331
5      137
6     1133
7      192
8      197
9      172
10     460
11     231
12     474
Name: weight_dvdl, dtype: int64
Lambda
1     235
2     188
3     153
4     186
5     211
6     332
7     458
8     749
9     133
10    507
11    385
12    253
Name: weight_dvdl, dtype: int64
Lambda
1     168
2     155
3     182
4     106
5     611
6     801
7     149
8     175
9     228
10    347
11    386
12    317
Name: weight_dvdl, dtype: int64


In [31]:
st1_vdw_dGs = np.array(bootstrap_df(st1_vdw_for_dG, 1000, 200))
st2_vdw_dGs = np.array(bootstrap_df(st2_vdw_boost, 1000, 200))
st3_vdw_dGs = np.array(bootstrap_df(st3_vdw_boost, 1000, 200))

a1vdw = f71c2_d28c1[f71c2_d28c1["Y"] < 100]["pop"].sum()
a2vdw = f71c2_d28c1[(f71c2_d28c1["Y"] > 120) & (f71c2_d28c1["Y"] < 220)]["pop"].sum()
a3vdw = f71c2_d28c1[f71c2_d28c1["Y"] > 220]["pop"].sum()

a1_prop_vdw = a1vdw/(a1vdw + a2vdw + a3vdw)
a2_prop_vdw = a2vdw/(a1vdw + a2vdw + a3vdw)
a3_prop_vdw = a3vdw/(a1vdw + a2vdw + a3vdw)

corr_vdw_dGs = a1_prop_vdw * st1_vdw_dGs + a2_prop_vdw * st2_vdw_dGs + a3_prop_vdw * st3_vdw_dGs

#### Incorporating vdw step correction (unbound state)

We will concisely reproduce the correction here, but for more detail please see the `R66Gvdw_example.ipynb` notebook. 

In [32]:
# shifting coordinates for simplicity (-180, 180) --> (0, 360)
geom_dvdls_vdw_ub.loc[geom_dvdls_vdw_ub["D248_chi1"] < 0, "D248_chi1"] += 360

# filtering TI sampling to match GaMD pmf 
geom_dvdls_vdw_ub["in_pmf_cont"] = rot_in_pmf_cont(geom_dvdls_vdw_ub, "F291_chi2", "D248_chi1", f71c2_d28c1_ub, 10)

st1ub_vdw = geom_dvdls_vdw_ub[
    (geom_dvdls_vdw_ub["in_pmf_cont"]) & 
    (geom_dvdls_vdw_ub["D248_chi1"] > 0) &
    (geom_dvdls_vdw_ub["D248_chi1"] < 100) &
    (geom_dvdls_vdw_ub["F291_chi2"] < 0)
]

st2ub_vdw = geom_dvdls_vdw_ub[
    (geom_dvdls_vdw_ub["in_pmf_cont"]) & 
    (geom_dvdls_vdw_ub["D248_chi1"] > 120) &
    (geom_dvdls_vdw_ub["D248_chi1"] < 220) &
    (geom_dvdls_vdw_ub["F291_chi2"] < 0)
]

st3ub_vdw = geom_dvdls_vdw_ub[
    (geom_dvdls_vdw_ub["in_pmf_cont"]) & 
    (geom_dvdls_vdw_ub["D248_chi1"] > 250) &
    (geom_dvdls_vdw_ub["D248_chi1"] < 350) & 
    (geom_dvdls_vdw_ub["F291_chi2"] < 0)
]

st4ub_vdw = geom_dvdls_vdw_ub[
    (geom_dvdls_vdw_ub["in_pmf_cont"]) & 
    (geom_dvdls_vdw_ub["D248_chi1"] > 0) &
    (geom_dvdls_vdw_ub["D248_chi1"] < 100) &
    (geom_dvdls_vdw_ub["F291_chi2"] > 0)
]

st5ub_vdw = geom_dvdls_vdw_ub[
    (geom_dvdls_vdw_ub["in_pmf_cont"]) & 
    (geom_dvdls_vdw_ub["D248_chi1"] > 120) &
    (geom_dvdls_vdw_ub["D248_chi1"] < 220) &
    (geom_dvdls_vdw_ub["F291_chi2"] > 0)
]

st6ub_vdw = geom_dvdls_vdw_ub[
    (geom_dvdls_vdw_ub["in_pmf_cont"]) & 
    (geom_dvdls_vdw_ub["D248_chi1"] > 250) &
    (geom_dvdls_vdw_ub["D248_chi1"] < 350) & 
    (geom_dvdls_vdw_ub["F291_chi2"] > 0)
]

print("Microstate 1: ")
print(st1ub_vdw.groupby("Lambda").count()["weight_dvdl"])
print("Microstate 2: ")
print(st2ub_vdw.groupby("Lambda").count()["weight_dvdl"])
print("Microstate 3: ")
print(st3ub_vdw.groupby("Lambda").count()["weight_dvdl"])
print("Microstate 4: ")
print(st4ub_vdw.groupby("Lambda").count()["weight_dvdl"])
print("Microstate 5: ")
print(st5ub_vdw.groupby("Lambda").count()["weight_dvdl"])
print("Microstate 6: ")
print(st6ub_vdw.groupby("Lambda").count()["weight_dvdl"])


Microstate 1: 
Series([], Name: weight_dvdl, dtype: int64)
Microstate 2: 
Lambda
1      1
3      1
4      9
5     11
6     54
7     19
9     15
10     1
12    25
Name: weight_dvdl, dtype: int64
Microstate 3: 
Lambda
1     213
2     228
3     257
4     273
5     242
6     236
7     199
8     255
9     301
10    320
11    326
12    309
Name: weight_dvdl, dtype: int64
Microstate 4: 
Lambda
1     10
2     57
3      1
5      4
6      3
7     57
8      6
9     10
10    11
12     2
Name: weight_dvdl, dtype: int64
Microstate 5: 
Lambda
1     252
2     274
3     247
4     224
5     213
6     186
7      25
8      68
9     128
10     26
11     55
12     76
Name: weight_dvdl, dtype: int64
Microstate 6: 
Lambda
1       3
2      37
3      19
4      54
5      18
6      62
7     192
8     207
9     126
10    253
11    205
12    205
Name: weight_dvdl, dtype: int64


#### Ingesting more sampling to fill in missing lambdas, unbound state

In [33]:
os.chdir("../../TI_data/VL-R66G")
geom_dvdls_vdw_boost_ub = pd.read_csv("R66G_vdw_boost_ub.csv")

geom_dvdls_vdw_boost_ub.loc[geom_dvdls_vdw_boost_ub["D248_chi1"] < 0, "D248_chi1"] += 360
geom_dvdls_vdw_boost_ub["in_pmf_cont"] = rot_in_pmf_cont(geom_dvdls_vdw_boost_ub, "F291_chi2", "D248_chi1", f71c2_d28c1_ub, 10)

In [34]:
st1ub_vdw_missing = geom_dvdls_vdw_boost_ub[
    (geom_dvdls_vdw_boost_ub["in_pmf_cont"]) & 
    (geom_dvdls_vdw_boost_ub["D248_chi1"] > 0) &
    (geom_dvdls_vdw_boost_ub["D248_chi1"] < 100) &
    (geom_dvdls_vdw_boost_ub["F291_chi2"] < 0) &
    (geom_dvdls_vdw_boost_ub["Lambda"].isin(range(1, 13)))
]

st2ub_vdw_missing = geom_dvdls_vdw_boost_ub[
    (geom_dvdls_vdw_boost_ub["in_pmf_cont"]) & 
    (geom_dvdls_vdw_boost_ub["D248_chi1"] > 120) &
    (geom_dvdls_vdw_boost_ub["D248_chi1"] < 220) &
    (geom_dvdls_vdw_boost_ub["F291_chi2"] < 0) & 
    (geom_dvdls_vdw_boost_ub["Lambda"].isin(range(1, 13)))
]


st4ub_vdw_missing = geom_dvdls_vdw_boost_ub[
    (geom_dvdls_vdw_boost_ub["in_pmf_cont"]) & 
    (geom_dvdls_vdw_boost_ub["D248_chi1"] > 0) &
    (geom_dvdls_vdw_boost_ub["D248_chi1"] < 100) &
    (geom_dvdls_vdw_boost_ub["F291_chi2"] > 0) & 
    (geom_dvdls_vdw_boost_ub["Lambda"].isin(range(1, 13)))
]

st5ub_vdw_missing = geom_dvdls_vdw_boost_ub[
    (geom_dvdls_vdw_boost_ub["in_pmf_cont"]) & 
    (geom_dvdls_vdw_boost_ub["D248_chi1"] > 120) &
    (geom_dvdls_vdw_boost_ub["D248_chi1"] < 220) &
    (geom_dvdls_vdw_boost_ub["F291_chi2"] > 0) & 
    (geom_dvdls_vdw_boost_ub["Lambda"].isin([7,8,10,11,12]))
]

st6ub_vdw_missing = geom_dvdls_vdw_boost_ub[
    (geom_dvdls_vdw_boost_ub["in_pmf_cont"]) & 
    (geom_dvdls_vdw_boost_ub["D248_chi1"] > 250) &
    (geom_dvdls_vdw_boost_ub["D248_chi1"] < 350) & 
    (geom_dvdls_vdw_boost_ub["F291_chi2"] > 0) & 
    (geom_dvdls_vdw_boost_ub["Lambda"].isin(range(1, 7)))
]

st1ub_vdw_boost = pd.concat([st1ub_vdw, st1ub_vdw_missing])
st2ub_vdw_boost = pd.concat([st2ub_vdw, st2ub_vdw_missing])
st3ub_vdw_boost = st3ub_vdw
st4ub_vdw_boost = pd.concat([st4ub_vdw, st4ub_vdw_missing])
st5ub_vdw_boost = pd.concat([st5ub_vdw, st5ub_vdw_missing])
st6ub_vdw_boost = pd.concat([st6ub_vdw, st6ub_vdw_missing])


print("Microstate 1: ")
print(st1ub_vdw_boost.groupby("Lambda").count()["weight_dvdl"])
print("Microstate 2: ")
print(st2ub_vdw_boost.groupby("Lambda").count()["weight_dvdl"])
print("Microstate 3: ")
print(st3ub_vdw_boost.groupby("Lambda").count()["weight_dvdl"])
print("Microstate 4: ")
print(st4ub_vdw_boost.groupby("Lambda").count()["weight_dvdl"])
print("Microstate 5: ")
print(st5ub_vdw_boost.groupby("Lambda").count()["weight_dvdl"])
print("Microstate 6: ")
print(st6ub_vdw_boost.groupby("Lambda").count()["weight_dvdl"])


Microstate 1: 
Lambda
4     407
5     168
10    236
12     16
Name: weight_dvdl, dtype: int64
Microstate 2: 
Lambda
1      41
2      45
3     708
4     226
5     586
6     341
7     154
8      75
9      94
10     87
11    508
12    444
Name: weight_dvdl, dtype: int64
Microstate 3: 
Lambda
1     213
2     228
3     257
4     273
5     242
6     236
7     199
8     255
9     301
10    320
11    326
12    309
Name: weight_dvdl, dtype: int64
Microstate 4: 
Lambda
1      660
2     1171
3       17
4      376
5     1096
6      726
7      213
8       63
9      218
10     117
11     315
12      15
Name: weight_dvdl, dtype: int64
Microstate 5: 
Lambda
1     252
2     274
3     247
4     224
5     213
6     186
7     459
8     420
9     128
10    412
11    421
12    480
Name: weight_dvdl, dtype: int64
Microstate 6: 
Lambda
1     399
2      63
3     683
4     285
5     259
6     409
7     192
8     207
9     126
10    253
11    205
12    205
Name: weight_dvdl, dtype: int64


#### Filling in any remaining missing values by aggregating the rest of the microstates

In [35]:
for_st1ub_vdw = pd.concat([st2ub_vdw_boost, st3ub_vdw_boost, st4ub_vdw_boost, st5ub_vdw_boost, st6ub_vdw_boost])
for_st2ub_vdw = pd.concat([st1ub_vdw_boost, st3ub_vdw_boost, st4ub_vdw_boost, st5ub_vdw_boost, st6ub_vdw_boost])
for_st4ub_vdw = pd.concat([st1ub_vdw_boost, st2ub_vdw_boost, st3ub_vdw_boost, st5ub_vdw_boost, st6ub_vdw_boost])
for_st6ub_vdw = pd.concat([st1ub_vdw_boost, st2ub_vdw_boost, st3ub_vdw_boost, st4ub_vdw_boost, st5ub_vdw_boost])

st1ub_vdw_missing = for_st1ub_vdw[for_st1ub_vdw["Lambda"].isin([1,2,3,6,7,8,9,11,12])]
st2ub_vdw_missing = for_st2ub_vdw[for_st2ub_vdw["Lambda"].isin([1,2,8,9,10])]
st4ub_vdw_missing = for_st4ub_vdw[for_st4ub_vdw["Lambda"].isin([3,8,12])]
st6ub_vdw_missing = for_st6ub_vdw[for_st6ub_vdw["Lambda"].isin([2])]

st1ub_vdw_for_dG = pd.concat([st1ub_vdw_boost, st1ub_vdw_missing])
st2ub_vdw_for_dG = pd.concat([st2ub_vdw_boost, st2ub_vdw_missing])
st3ub_vdw_for_dG = st3ub_vdw_boost
st4ub_vdw_for_dG = pd.concat([st4ub_vdw_boost, st4ub_vdw_missing])
st5ub_vdw_for_dG = st5ub_vdw_boost
st6ub_vdw_for_dG = pd.concat([st6ub_vdw_boost, st6ub_vdw_missing])

print("Microstate 1: ")
print(st1ub_vdw_for_dG.groupby("Lambda").count()["weight_dvdl"])
print("Microstate 2: ")
print(st2ub_vdw_for_dG.groupby("Lambda").count()["weight_dvdl"])
print("Microstate 3: ")
print(st3ub_vdw_for_dG.groupby("Lambda").count()["weight_dvdl"])
print("Microstate 4: ")
print(st4ub_vdw_for_dG.groupby("Lambda").count()["weight_dvdl"])
print("Microstate 5: ")
print(st5ub_vdw_for_dG.groupby("Lambda").count()["weight_dvdl"])
print("Microstate 6: ")
print(st6ub_vdw_for_dG.groupby("Lambda").count()["weight_dvdl"])


Microstate 1: 
Lambda
1     1565
2     1781
3     1912
4      407
5      168
6     1898
7     1217
8     1020
9      867
10     236
11    1775
12    1469
Name: weight_dvdl, dtype: int64
Microstate 2: 
Lambda
1     1565
2     1781
3      708
4      226
5      586
6      341
7      154
8     1020
9      867
10    1425
11     508
12     444
Name: weight_dvdl, dtype: int64
Microstate 3: 
Lambda
1     213
2     228
3     257
4     273
5     242
6     236
7     199
8     255
9     301
10    320
11    326
12    309
Name: weight_dvdl, dtype: int64
Microstate 4: 
Lambda
1      660
2     1171
3     1912
4      376
5     1096
6      726
7      213
8     1020
9      218
10     117
11     315
12    1469
Name: weight_dvdl, dtype: int64
Microstate 5: 
Lambda
1     252
2     274
3     247
4     224
5     213
6     186
7     459
8     420
9     128
10    412
11    421
12    480
Name: weight_dvdl, dtype: int64
Microstate 6: 
Lambda
1      399
2     1781
3      683
4      285
5      259
6      409
7     

#### Computing microstate areas and final corrected dG of unbound state

In [36]:
a1ub_vdw = f71c2_d28c1_ub[
    (f71c2_d28c1_ub["X"] < 0) & 
    (f71c2_d28c1_ub["Y"] < 100)
]["pop"].sum()

a2ub_vdw = f71c2_d28c1_ub[
    (f71c2_d28c1_ub["X"] < 0) & 
    (f71c2_d28c1_ub["Y"] > 100) & 
    (f71c2_d28c1_ub["Y"] < 220)
]["pop"].sum()

a3ub_vdw = f71c2_d28c1_ub[
    (f71c2_d28c1_ub["X"] < 0) & 
    (f71c2_d28c1_ub["Y"] > 220) & 
    (f71c2_d28c1_ub["Y"] < 350)
]["pop"].sum()

a4ub_vdw = f71c2_d28c1_ub[
    (f71c2_d28c1_ub["X"] > 0) & 
    (f71c2_d28c1_ub["Y"] < 100)
]["pop"].sum()

a5ub_vdw = f71c2_d28c1_ub[
    (f71c2_d28c1_ub["X"] > 0) & 
    (f71c2_d28c1_ub["Y"] > 100) & 
    (f71c2_d28c1_ub["Y"] < 220)
]["pop"].sum()

a6ub_vdw = f71c2_d28c1_ub[
    (f71c2_d28c1_ub["X"] > 0) & 
    (f71c2_d28c1_ub["Y"] > 220) & 
    (f71c2_d28c1_ub["Y"] < 350)
]["pop"].sum()

ub_areas_vdw = [a1ub_vdw, a2ub_vdw, a3ub_vdw, a4ub_vdw, a5ub_vdw, a6ub_vdw]
ub_area_props_vdw = np.array([i/sum(ub_areas_vdw) for i in ub_areas_vdw])

state_dGs_ub_vdw = [
    np.array(bootstrap_df(st1ub_vdw_for_dG, 1000, 200)),
    np.array(bootstrap_df(st2ub_vdw_for_dG, 1000, 200)),
    np.array(bootstrap_df(st3ub_vdw_for_dG, 1000, 200)),
    np.array(bootstrap_df(st4ub_vdw_for_dG, 1000, 200)),
    np.array(bootstrap_df(st5ub_vdw_for_dG, 1000, 200)),
    np.array(bootstrap_df(st6ub_vdw_for_dG, 1000, 200))
]

corr_vdw_ub_dGs = np.dot(ub_area_props_vdw, state_dGs_ub_vdw)

#### Charging + vdw step correction results

With the charging step correction based on the VL-N30 chi1/chi2 rotamer sampling, there is negligible change in the charging step ddG (~0.1 kcal/mol increase).

With the vdw step correction based on VL-F71 chi2 and VL-D28 chi1 rotamer sampling, there is a +0.65 kcal/mol increase in the vdw step ddG.

In [37]:
print("Values after corrections to both crg step and vdw step")

corr_ddGs_fin = print_summary_crg(corr_crg_dGs, corr_crg_ub_dGs, corr_vdw_dGs, corr_vdw_ub_dGs)

corr_err = abs(empirical_value - np.mean(corr_ddGs_fin))
print()
print(f"\nEmpirical value: {empirical_value} kcal/mol")
print(f"Error after corrections: {round(corr_err, 4)} kcal/mol")

Values after corrections to both crg step and vdw step
Charging step ddG: -1.0133 +/- 0.6596
Lower CI: -1.6729
Upper CI: -0.3755

Vdw step ddG: 0.7934 +/- 0.3303
Lower CI: 0.4658
Upper CI: 1.1237

Total ddG: -0.2199 +/- 0.704
Lower CI: -0.9148
Upper CI: 0.4841


Empirical value: 0.22 kcal/mol
Error after corrections: 0.4399 kcal/mol


#### Original values for comparison

In [38]:
total_ddGs = print_summary_crg(orig_crg_dGs, orig_crg_ub_dGs, orig_vdw_dGs, orig_vdw_ub_dGs)
print(f"\nEmpirical value: {empirical_value} kcal/mol")
print(f"Original error: {round(orig_error, 4)} kcal/mol")

Charging step ddG: -0.8542 +/- 0.8496
Lower CI: -1.7038
Upper CI: -0.0106

Vdw step ddG: 0.1512 +/- 0.6432
Lower CI: -0.4655
Upper CI: 0.7944

Total ddG: -0.703 +/- 1.0762
Lower CI: -1.7437
Upper CI: 0.3732

Empirical value: 0.22 kcal/mol
Original error: 0.923 kcal/mol
